## `nz_splines_magabs.ipynb`
-------------

In [ ]:
import numpy as np
import importlib
import json
import matplotlib.pyplot as plt

from pathlib import Path

import src.statistics.spline as spline
import src.statistics.corrfiles as cf
import src.statistics.systematics as sy

importlib.reload(spline)
importlib.reload(sy)

ROOT = cf.get_base_dir()

In [ ]:
STUDY = "magabs"

scale_cut = [0.3, 3]
version = "v_1p1"
name = "npz_bs_bp_mag"

tag = sy.scale_cut_tag(scale_cut)
DATA_DIR = sy.variant_dir(ROOT, STUDY, scale_cut, version)
SPL_DIR = sy.variant_dir(ROOT, STUDY, scale_cut, version, what="splines")
SPL_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / f"{STUDY}_metadata_{tag}_{version}.json") as f:
    meta = json.load(f)
N_REALIZATIONS = meta["n_realizations"]

FIT_KWARGS = sy.SPLINE_FIT_KWARGS

print(f"alpha perturbation : {meta['mag_perturbation']} ({meta.get('mode')})")
print(f"Realizations       : 0 (unperturbed) + {N_REALIZATIONS}")
print(f"Data   : {DATA_DIR}")
print(f"Splines: {SPL_DIR}")

In [ ]:
failed = []
for r in range(N_REALIZATIONS + 1):
    data = np.load(DATA_DIR / f"merged_res_norm_{tag}_{version}_r{r:02d}.npz")
    for tomo in sy.TOMO_BINS:
        savefile = str(SPL_DIR / f"spl_{name}_{tomo}_r{r:02d}")
        if Path(f"{savefile}.nc").exists():
            print(f"Skipping {savefile}, already exists")
            continue

        z = data[f"{tomo}/{name}_z"]
        npz_arr = data[f"{tomo}/{name}"]
        npz_arr_err = data[f"{tomo}/{name}_err"]

        print(f"\n=== realization {r:02d}, tomo {tomo} ({len(z)} points) ===")
        try:
            spl = spline.BayesianBSpline(zv=z, n_knots=int(len(z) // 2))
            spl.fit(npz_arr, npz_arr_err, **FIT_KWARGS)
            spl.save_model(savefile)
        except Exception as exc:  # noqa: BLE001 -- keep the remaining fits going
            failed.append((r, tomo, repr(exc)))
            print(f"  FAILED realization {r:02d}, tomo {tomo}: {exc!r}")

if failed:
    print(f"\n{len(failed)} fit(s) failed; re-run this cell to retry just those:")
    for r, tomo, exc in failed:
        print(f"  realization {r:02d}, tomo {tomo}: {exc}")
else:
    print("\nAll fits complete.")

In [ ]:
fid_file = ROOT / "results" / f"splines_{tag}_{version}" / f"spl_{name}_1"
if Path(f"{fid_file}.nc").exists():
    fid = spline.BayesianBSpline.from_saved_model(str(fid_file))
    r0 = spline.BayesianBSpline.from_saved_model(str(SPL_DIR / f"spl_{name}_1_r00"))
    print("max |dn(z)| between fiducial and realization 0 inputs :",
          np.abs(fid.nz - r0.nz).max())
    print("max |dsigma| :", np.abs(fid.nz_err - r0.nz_err).max())
else:
    print(f"No fiducial spline at {fid_file}; run nz_splines.ipynb first to compare.")

In [ ]:
z_all = np.linspace(0, 3, 600)
fig, axs = plt.subplots(2, 2, figsize=(11, 7))
for tomo, ax in zip(sy.TOMO_BINS, axs.flat):
    for r in range(N_REALIZATIONS + 1):
        f = SPL_DIR / f"spl_{name}_{tomo}_r{r:02d}"
        if not Path(f"{f}.nc").exists():
            continue
        spl = spline.BayesianBSpline.from_saved_model(str(f))
        mask = (z_all <= spl.zv.max()) & (z_all >= spl.zv.min())
        samples = sy.normalized_samples(spl, z_all[mask])
        ax.plot(
            z_all[mask],
            np.percentile(samples, 50, axis=0),
            color="crimson" if r == 0 else "gray",
            lw=2 if r == 0 else 0.8,
            alpha=1 if r == 0 else 0.6,
            zorder=5 if r == 0 else 1,
        )
    ax.set_title(f"Bin {tomo}")
    ax.set_xlabel("Redshift")
    ax.set_ylabel("n(z)")
    ax.grid(True, alpha=0.3)
fig.suptitle("Spline medians: unperturbed (red) vs perturbed realizations (grey)")
fig.tight_layout()